# Percorso completo: dalla pulizia dati al classificatore di memoria

**Cos'è questo notebook.** Una sintesi eseguibile delle Lezioni 1-15 del corso,
dall'importazione dei dati grezzi fino al modello finale che classifica il
tipo di una memoria (`episodic`, `preference`, `semantic`, `unknown`) a
partire dal suo testo. Ogni sezione qui sotto condensa il codice essenziale
di 2-4 lezioni — le spiegazioni teoriche complete, gli esercizi e le
dimostrazioni isolate restano nei notebook originali (`lezione-01-...` a
`lezione-15-...`), a cui questo file rimanda.

**Perché esiste.** Per vedere l'intera pipeline di dati → feature → modello
girare in un colpo solo, dall'inizio alla fine, e per avere in un solo posto
una stima delle risorse di calcolo necessarie per eseguirla (ultima sezione).

**Cosa NON è.** Non sostituisce le 15 lezioni: quelle insegnano *perché* ogni
passo è fatto in quel modo, con esercizi guidati. Questo notebook mostra
*cosa* succede quando la pipeline gira per intero, senza le pause didattiche.

**Stato del percorso.** Le Lezioni 1-15 (Fase 0 fondamenti, Fase 1 pulizia
dati, Fase 2 reti dense Keras, inizio Fase 3 testo) sono complete e sono
quello che questo notebook riproduce. Le fasi successive del corso
(embedding, rappresentazione delle memorie, Transformer/Gemma, LoRA,
pipeline, capstone) sono ancora pianificate — vedi `course/course.yaml` — e
non sono ancora incluse qui.

## Fase 1 — Dati puliti (Lezioni 1-5)

Tre batch di memorie grezze arrivano in momenti diversi (come farebbero in
produzione). Applichiamo in sequenza: gestione dei valori mancanti
(Lezione 1), deduplicazione e correzione di tipi/outlier (Lezione 2),
split train/validation/test ordinato nel tempo (Lezione 3), audit
anti-leakage (Lezione 4), estrazione ed encoding delle feature (Lezione 5).

In [ ]:
import time
import pandas as pd
from pathlib import Path

_t0_notebook = time.perf_counter()  # per la stima del tempo totale, ultima cella

synthetic = Path('..') / 'datasets' / 'synthetic'
processed = Path('..') / 'datasets' / 'processed'
processed.mkdir(parents=True, exist_ok=True)

# --- Lezione 1: valori mancanti -------------------------------------------
memorie = pd.read_csv(synthetic / 'memory_events_raw.csv')
CRITICI = ['memory_id', 'text', 'timestamp']
memorie = memorie.dropna(subset=CRITICI).reset_index(drop=True)

memorie['type_was_missing'] = memorie['type'].isna()
memorie['type'] = memorie['type'].fillna('unknown')
memorie['importance_was_missing'] = memorie['importance'].isna()
memorie['importance'] = memorie['importance'].fillna(memorie['importance'].median())

print(f'Batch 1 (raw): {len(memorie)} memorie dopo la pulizia dei valori mancanti')

In [ ]:
# --- Lezione 2: duplicati, tipi, outlier -----------------------------------
nuovo_batch = pd.read_csv(synthetic / 'memory_events_quality_issues.csv')
memorie = pd.concat([memorie, nuovo_batch], ignore_index=True)

testo_norm = memorie['text'].astype('string').str.strip().str.casefold()
candidati = (
    memorie.duplicated('memory_id', keep='first')
    | memorie.duplicated(['text', 'timestamp'], keep='first')
    | memorie.assign(_n=testo_norm).duplicated(['_n', 'timestamp'], keep='first')
)
memorie = memorie.loc[~candidati].reset_index(drop=True)

numeri = pd.to_numeric(memorie['importance'], errors='coerce')
memorie['importance_was_invalid_type'] = memorie['importance'].notna() & numeri.isna()
memorie['importance'] = numeri.fillna(numeri[numeri.between(0, 1)].median())
memorie['importance_was_outlier'] = ~memorie['importance'].between(0, 1)
memorie['importance'] = memorie['importance'].clip(0, 1)

for flag in ['type_was_missing', 'importance_was_missing']:
    memorie[flag] = memorie[flag].fillna(False)
memorie['type'] = memorie['type'].fillna('unknown')

memorie.to_csv(processed / 'memory_events_clean.csv', index=False)
print(f'Batch 2 (+ quality_issues): {len(memorie)} memorie dopo dedup e correzione tipi'
      f' ({int(candidati.sum())} duplicati rimossi)')

In [ ]:
# --- Lezione 3: split train/validation/test ordinato nel tempo ------------
storico = pd.read_csv(synthetic / 'memory_events_history.csv')
memorie = pd.concat([memorie, storico], ignore_index=True)

memorie = memorie.dropna(subset=CRITICI).reset_index(drop=True)
memorie['type'] = memorie['type'].fillna('unknown')
numeri = pd.to_numeric(memorie['importance'], errors='coerce')

testo_norm = memorie['text'].astype('string').str.strip().str.casefold()
doppi = (
    memorie.duplicated('memory_id', keep='first')
    | memorie.assign(_n=testo_norm).duplicated(['_n', 'timestamp'], keep='first')
)
memorie = memorie.loc[~doppi].reset_index(drop=True)
numeri = numeri.loc[memorie.index]
memorie['importance'] = numeri.fillna(numeri[numeri.between(0, 1)].median()).clip(0, 1)

memorie['timestamp'] = pd.to_datetime(memorie['timestamp'], format='mixed')
memorie = memorie.sort_values('timestamp').reset_index(drop=True)

n = len(memorie)
fine_train, fine_val = int(n * 0.70), int(n * 0.85)
mem_train = memorie.iloc[:fine_train]
mem_val = memorie.iloc[fine_train:fine_val]
mem_test = memorie.iloc[fine_val:]

assert mem_train['timestamp'].max() <= mem_val['timestamp'].min()
assert mem_val['timestamp'].max() <= mem_test['timestamp'].min()

mem_train.to_csv(processed / 'memory_train.csv', index=False)
mem_val.to_csv(processed / 'memory_val.csv', index=False)
mem_test.to_csv(processed / 'memory_test.csv', index=False)
print(f'Batch 3 (+ history): {n} memorie totali dopo dedup')
print(f'split -> train {len(mem_train)} | val {len(mem_val)} | test {len(mem_test)}')

In [ ]:
# --- Lezione 4: audit anti-leakage -----------------------------------------
mem_train = pd.read_csv(processed / 'memory_train.csv')
mem_val = pd.read_csv(processed / 'memory_val.csv')
mem_test = pd.read_csv(processed / 'memory_test.csv')

id_tr, id_va, id_te = set(mem_train.memory_id), set(mem_val.memory_id), set(mem_test.memory_id)
assert not (id_tr & id_va) and not (id_tr & id_te) and not (id_va & id_te)

def testi_norm(frame):
    return set(frame['text'].astype('string').str.strip().str.casefold())

contaminati = testi_norm(mem_train)
sovrapposti_val = contaminati & testi_norm(mem_val)
mem_val = mem_val[~mem_val['text'].astype('string').str.strip().str.casefold().isin(contaminati)]
contaminati |= testi_norm(mem_val)
sovrapposti_test = contaminati & testi_norm(mem_test)
mem_test = mem_test[~mem_test['text'].astype('string').str.strip().str.casefold().isin(contaminati)]

mem_val.to_csv(processed / 'memory_val.csv', index=False)
mem_test.to_csv(processed / 'memory_test.csv', index=False)
print(f'Testi contaminati rimossi da val: {len(sovrapposti_val)}, da test: {len(sovrapposti_test)}')
print(f'dopo audit -> train {len(mem_train)} | val {len(mem_val)} | test {len(mem_test)}')

In [ ]:
# --- Lezione 5: estrazione feature, encoding, scaling ----------------------
insiemi = {n: pd.read_csv(processed / f'memory_{n}.csv') for n in ['train', 'val', 'test']}

def estrai_feature(frame):
    out = pd.DataFrame(index=frame.index)
    testo = frame['text'].astype('string')
    out['text_length'] = testo.str.len()
    out['word_count'] = testo.str.split().str.len()
    out['importance'] = frame['importance']
    for flag in ['type_was_missing', 'importance_was_missing',
                 'importance_was_invalid_type', 'importance_was_outlier']:
        out[flag] = frame.get(flag, pd.Series(False, index=frame.index)).fillna(False).astype(int)
    return out

feature = {n: estrai_feature(f) for n, f in insiemi.items()}
classi = sorted(insiemi['train']['type'].unique())
mappa_classi = {c: i for i, c in enumerate(classi)}

numeriche = ['text_length', 'word_count', 'importance']
media, dev = feature['train'][numeriche].mean(), feature['train'][numeriche].std()
for n in feature:
    feature[n][numeriche] = (feature[n][numeriche] - media) / dev
    feature[n]['target'] = insiemi[n]['type'].map(mappa_classi)
    feature[n].to_csv(processed / f'memory_features_{n}.csv', index=False)

print('Classi:', mappa_classi)
for n, f in feature.items():
    print(f'  memory_features_{n}.csv: {f.shape[0]} righe x {f.shape[1]} colonne')

## Fase 0 — Fondamenti: array, gradiente, loss (Lezioni 6-9)

Prima di usare Keras, costruiamo un classificatore "a mano" con NumPy puro:
matrici, discesa del gradiente, softmax e cross-entropy. Il risultato è
l'**asticella di riferimento** (baseline) che ogni rete Keras successiva
dovrà battere.

In [ ]:
# --- Lezione 6-7: array NumPy, vettorizzazione, prodotto matrice-vettore ---
import numpy as np
import time

# vettorizzazione: stessa operazione, due modi
dati = list(range(200_000))
array = np.arange(200_000)
t0 = time.perf_counter()
_ = [x * x for x in dati]
t1 = time.perf_counter()
_ = array * array
t2 = time.perf_counter()
print(f'ciclo Python: {(t1 - t0) * 1000:.1f} ms   vettorizzato: {(t2 - t1) * 1000:.1f} ms')

train_feat = pd.read_csv(processed / 'memory_features_train.csv')
val_feat = pd.read_csv(processed / 'memory_features_val.csv')
X_train = train_feat.drop(columns='target').to_numpy()
y_train = train_feat['target'].to_numpy()
X_val = val_feat.drop(columns='target').to_numpy()
y_val = val_feat['target'].to_numpy()

rng = np.random.default_rng(42)
W_casuale = rng.normal(0, 0.1, (X_train.shape[1], len(classi)))
predizioni_casuali = (X_train @ W_casuale).argmax(axis=1)
print(f'X: {X_train.shape}   pesi casuali: {W_casuale.shape}')
print(f'accuratezza con pesi CASUALI: {(predizioni_casuali == y_train).mean():.0%}'
      f'  (atteso vicino a 1/{len(classi)} = {1/len(classi):.0%})')

In [ ]:
# --- Lezione 8: discesa del gradiente (regressione 1D di prova) -----------
x_tr, t_tr = train_feat['text_length'].to_numpy(), train_feat['importance'].to_numpy()
x_va, t_va = val_feat['text_length'].to_numpy(), val_feat['importance'].to_numpy()

w, b, passo = 0.0, 0.0, 0.1
for epoca in range(200):
    errore = (w * x_tr + b) - t_tr
    w -= passo * 2 * (errore * x_tr).mean()
    b -= passo * 2 * errore.mean()

mse_val = np.mean((w * x_va + b - t_va) ** 2)
mse_banale = np.mean((t_tr.mean() - t_va) ** 2)
print(f'regressione 1D (text_length -> importance): w={w:+.3f}, b={b:+.3f}')
print(f'errore su validation: {mse_val:.4f}   baseline (media fissa): {mse_banale:.4f}')

In [ ]:
# --- Lezione 9: softmax + cross-entropy, addestrati a mano -----------------
def softmax(punteggi):
    e = np.exp(punteggi - punteggi.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

n_classi = len(classi)
Y_onehot = np.eye(n_classi)[y_train]
W = np.zeros((X_train.shape[1], n_classi))
b = np.zeros(n_classi)
passo = 0.5

for epoca in range(300):
    p = softmax(X_train @ W + b)
    grad_punteggi = (p - Y_onehot) / len(Y_onehot)
    W -= passo * X_train.T @ grad_punteggi
    b -= passo * grad_punteggi.sum(axis=0)

def accuratezza_softmax(X, y):
    return ((X @ W + b).argmax(axis=1) == y).mean()

baseline_train = accuratezza_softmax(X_train, y_train)
baseline_val = accuratezza_softmax(X_val, y_val)
print(f'softmax scritto a mano — accuratezza train: {baseline_train:.0%}   validation: {baseline_val:.0%}')

np.save(processed / 'softmax_baseline_W.npy', W)
np.save(processed / 'softmax_baseline_b.npy', b)
print('Baseline salvata: la Fase 2 (Keras) dovrà batterla.')

## Fase 2 — Reti dense con Keras (Lezioni 10-13)

Stesso problema, stessi dati, ora con Keras: prima una rete semplice, poi la
dimostrazione dal vivo dell'overfitting, infine l'architettura finale
(regolarizzata con Dropout ed Early Stopping) che diventa il modello
salvato del corso.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import keras

keras.utils.set_random_seed(42)
print('Keras', keras.__version__)

X_train_f = X_train.astype('float32')
X_val_f = X_val.astype('float32')

# --- Lezione 10: prima rete Keras, confrontata con la baseline Fase 0 -----
rete_semplice = keras.Sequential([
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(n_classi, activation='softmax'),
])
rete_semplice.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
rete_semplice.fit(X_train_f, y_train, epochs=150, validation_data=(X_val_f, y_val), verbose=0)
acc_keras_semplice = rete_semplice.evaluate(X_val_f, y_val, verbose=0)[1]
print(f'softmax a mano (Fase 0)   : {baseline_val:.0%} su validation')
print(f'prima rete Keras (16 relu): {acc_keras_semplice:.0%} su validation')

In [ ]:
# --- Lezione 12 (parte 1): overfitting in diretta --------------------------
import matplotlib.pyplot as plt

keras.utils.set_random_seed(42)
rete_esagerata = keras.Sequential([
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(n_classi, activation='softmax'),
])
rete_esagerata.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
storia = rete_esagerata.fit(X_train_f, y_train, epochs=300,
                             validation_data=(X_val_f, y_val), verbose=0)

print(f'parametri della rete esagerata: {rete_esagerata.count_params()}'
      f'   esempi di train: {len(y_train)}  <- molti più pesi che esempi')

fig, asse = plt.subplots(figsize=(7, 3))
asse.plot(storia.history['loss'], label='loss train')
asse.plot(storia.history['val_loss'], label='loss validation')
asse.set_xlabel('epoca')
asse.legend()
asse.set_title('Overfitting: il train scende, la validation risale')
plt.tight_layout()
plt.show()

In [ ]:
# --- Lezione 12 (parte 2): architettura finale, regolarizzata -------------
test_feat = pd.read_csv(processed / 'memory_features_test.csv')
X_test_f = test_feat.drop(columns='target').to_numpy().astype('float32')
y_test = test_feat['target'].to_numpy()

keras.utils.set_random_seed(42)
modello_finale = keras.Sequential([
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(n_classi, activation='softmax'),
])
modello_finale.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
storia_finale = modello_finale.fit(X_train_f, y_train, epochs=300,
                                    validation_data=(X_val_f, y_val),
                                    callbacks=[stop], verbose=0)

print(f'fermato dopo {len(storia_finale.history["loss"])} epoche (early stopping, max 300)\n')
print(f'{"insieme":10} {"baseline Fase 0":>16} {"rete finale":>12}')
for nome, Xf, yf in [('train', X_train_f, y_train), ('val', X_val_f, y_val), ('test', X_test_f, y_test)]:
    base = accuratezza_softmax(Xf, yf)
    keras_acc = modello_finale.evaluate(Xf, yf, verbose=0)[1]
    print(f'{nome:10} {base:16.0%} {keras_acc:12.0%}')

Path('..', 'models').mkdir(exist_ok=True)
modello_finale.save('../models/memory_type_classifier.keras')
print('\nModello salvato in models/memory_type_classifier.keras')

In [ ]:
# --- Lezione 13: valutare il classificatore in dettaglio -------------------
from sklearn.metrics import classification_report

probabilita_val = modello_finale.predict(X_val_f, verbose=0)
predizioni_val = probabilita_val.argmax(axis=1)

print(classification_report(y_val, predizioni_val, labels=range(n_classi),
                             target_names=classi, zero_division=0))

confidenza_media = probabilita_val.max(axis=1).mean()
accuratezza_val = (predizioni_val == y_val).mean()
print(f'confidenza media dichiarata: {confidenza_media:.0%}   accuratezza reale: {accuratezza_val:.0%}')
if confidenza_media > accuratezza_val + 0.05:
    print('-> il modello è sovraconfidente: le probabilità promettono più di quanto mantengano')
else:
    print('-> confidenza e accuratezza sono compatibili')

## Fase 3 (inizio) — Leggere il testo delle memorie (Lezioni 14-15)

Finora il modello ha visto solo feature numeriche derivate dal testo
(lunghezza, conteggio parole), non il testo stesso. Ultimo passo: costruire
una pipeline `tf.data`, poi vettorizzare il testo vero e proprio in un
bag-of-words e riaddestrare — per vedere quanto conta leggere il contenuto,
non solo misurarne la forma.

In [ ]:
# --- Lezione 14: pipeline tf.data (shuffle, batch, prefetch) --------------
import tensorflow as tf

# illustrazione rapida: un buffer di shuffle piccolo mescola poco
numeri = tf.data.Dataset.range(10)
print('shuffle(buffer=2) :', list(numeri.shuffle(2, seed=0).as_numpy_iterator()))
print('shuffle(buffer=10):', list(numeri.shuffle(10, seed=0).as_numpy_iterator()))

train_ds = (tf.data.Dataset.from_tensor_slices((X_train_f, y_train))
            .shuffle(len(y_train), seed=42)
            .batch(32)
            .prefetch(tf.data.AUTOTUNE))
val_ds = tf.data.Dataset.from_tensor_slices((X_val_f, y_val)).batch(32)
print('\npipeline tf.data pronta: stesso dataset delle feature numeriche, ora a batch')

In [ ]:
# --- Lezione 15: bag-of-words dal testo vero, il salto di prestazioni -----
from keras.layers import TextVectorization

memorie_testo = {n: pd.read_csv(processed / f'memory_{n}.csv') for n in ['train', 'val']}
testi = {n: f['text'].astype(str).to_numpy() for n, f in memorie_testo.items()}
target_testo = {n: f['type'].map(mappa_classi).to_numpy() for n, f in memorie_testo.items()}

vettorizzatore = TextVectorization(max_tokens=300, output_mode='multi_hot')
vettorizzatore.adapt(testi['train'])  # vocabolario dal SOLO train
X_testo = {n: vettorizzatore(t).numpy() for n, t in testi.items()}
print('bag of words:', X_testo['train'].shape, '(memorie x parole del vocabolario)')

keras.utils.set_random_seed(42)
lettore = keras.Sequential([
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(n_classi, activation='softmax'),
])
lettore.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lettore.fit(X_testo['train'], target_testo['train'], epochs=300,
            validation_data=(X_testo['val'], target_testo['val']),
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                                       restore_best_weights=True)],
            verbose=0)

acc_testo = lettore.evaluate(X_testo['val'], target_testo['val'], verbose=0)[1]
print(f'\nfeature povere (lunghezza/conteggio, Lezioni 10-13): {accuratezza_val:.0%} su validation')
print(f'leggendo il testo vero (bag of words, Lezione 15)   : {acc_testo:.0%} su validation')

## Risultato finale e prossimi passi

| Fase | Modello | Accuratezza su validation |
|---|---|---|
| Fase 0 (Lezione 9) | softmax scritto a mano, feature numeriche | vedi output sopra |
| Fase 2 (Lezioni 10-13) | rete Keras 32→dropout→4, feature numeriche | vedi output sopra |
| Fase 3 (Lezione 15) | stessa rete, bag-of-words dal testo | vedi output sopra |

Il salto più grande della pipeline non viene da una rete più grande o da più
epoche: viene dal **dare al modello accesso al contenuto testuale** invece
che a due numeri derivati da esso (lunghezza, conteggio parole). È il primo
indizio del perché il corso dedica le fasi successive a embedding e
rappresentazione del significato, non solo a reti più profonde.

**Cosa manca ancora (pianificato, non incluso qui).** Embedding layer e
similarità coseno, rappresentazione strutturata delle memorie (schema,
importanza nel tempo, grafo), Transformer/Gemma, LoRA, pipeline di
produzione, capstone finale — vedi `course/course.yaml` e
`docs/modules/index.md` per lo stato aggiornato di ciascuna fase.

## Risorse e costo stimato

Numeri misurati eseguendo questo stesso notebook end-to-end (due run
separate, per controllare la stabilità), non stime teoriche: **55-58
secondi** di tempo totale, **781 MB** di picco di RAM del processo Python,
su una macchina CPU-only (nessuna GPU disponibile in questa sessione — TF
lo segnala esplicitamente all'avvio, e non ne ha comunque bisogno qui). La
cella successiva ricalcola questi due numeri ogni volta che esegui il
notebook, così puoi confrontarli con l'hardware che stai usando tu.

- **Dati**: poche centinaia di righe in tutto (213 esempi di train + 20
  validation + 15 test dopo deduplicazione e audit anti-leakage) — pochi KB
  per file CSV. Trascurabile: qualunque laptop o istanza Colab gratuita lo
  gestisce senza sforzo.
- **RAM**: **781 MB di picco misurato** in questa sessione. Il collo di
  bottiglia non è il dataset (piccolissimo) ma l'overhead di import di
  TensorFlow/Keras stesso, che pesa qualche centinaio di MB non appena la
  libreria viene caricata, indipendentemente da quanto è piccolo il modello
  addestrato dopo. Un ambiente con **almeno 1.5-2 GB di RAM libera** è un
  margine comodo rispetto al numero misurato.
- **GPU**: non necessaria, e in questa sessione non disponibile (CPU-only).
  I modelli addestrati qui hanno poche centinaia o migliaia di parametri
  (388 per il modello su feature numeriche, 2.276 per quello su
  bag-of-words) e i dataset stanno in poche centinaia di righe —
  l'addestramento gira comodamente su CPU in pochi secondi per modello. Una
  GPU non velocizzerebbe percepibilmente nulla a questa scala; il tempo
  dominante è l'overhead di avvio di TensorFlow, non il calcolo.
- **Disco**: l'ambiente Python con TensorFlow/Keras installati occupa circa
  **3 GB** (la libreria stessa, non i dati del corso — misurato: `.venv` a
  2.9 GB in questo ambiente). I dati e i modelli del corso sommati pesano
  poche centinaia di KB (il modello Keras salvato è ~28 KB).
- **Tempo di esecuzione**: **55-58 secondi** misurati end-to-end su CPU,
  dalla pulizia dati al modello finale sul testo vero. La parte più lenta
  non è l'addestramento dei singoli modelli (pochi secondi ciascuno su un
  dataset così piccolo) ma il primo `import keras`, che inizializza il
  backend TensorFlow — un costo fisso quasi indipendente da quanto lavoro
  fa poi il notebook.
- **Dove eseguirlo**: un runtime Colab gratuito (CPU) o un ambiente locale
  con `uv sync` sono più che sufficienti — non serve un runtime Colab Pro
  né un acceleratore a pagamento per questo notebook.

In [ ]:
# Misura effettiva di questa esecuzione (utile per verificare le stime sopra)
import resource
import sys

picco_ram_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024  # KB -> MB su Linux
tempo_totale_s = time.perf_counter() - _t0_notebook
print(f'Tempo totale di esecuzione di questo notebook: {tempo_totale_s:.0f} secondi')
print(f'Picco di RAM di questo processo Python: {picco_ram_mb:.0f} MB')
print(f'Parametri del modello finale (feature numeriche): {modello_finale.count_params()}')
print(f'Parametri del modello finale (bag-of-words testo): {lettore.count_params()}')
print(f'Versione Python: {sys.version.split()[0]}   Keras: {keras.__version__}')